In [1]:
# Install required libraries
!pip install transformers datasets accelerate -q

In [2]:
import torch
import math
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed
)
from datasets import Dataset

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

Using device: cuda


In [3]:
tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")   # lighter than GPT2
model = GPT2LMHeadModel.from_pretrained("distilgpt2")

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
def generate_text(model, tokenizer, prompt, max_length=60):
    model.eval()

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [5]:
review_prompts = [
    "This product is",
    "I bought this phone and",
    "The quality of this item"
]

baseline = {}

print("=== BASELINE OUTPUT ===\n")

for prompt in review_prompts:
    text = generate_text(model, tokenizer, prompt)
    baseline[prompt] = text
    print("Prompt:", prompt)
    print("Output:", text)
    print()

=== BASELINE OUTPUT ===

Prompt: This product is
Output: This product is designed to be simple to use for any occasion.
















































Prompt: I bought this phone and
Output: I bought this phone and I can't wait. I'll put a 5 star review on it. Will be buying the phone again in the next couple months. I really recommend it.
























Prompt: The quality of this item
Output: The quality of this item is not affected by the item and not necessarily reflect the quality of this item.










































In [6]:
corpus = [
    "this phone has an amazing battery life and the camera quality is outstanding for the price.",
    "i bought this laptop for college and it handles all my assignments and coding projects perfectly.",
    "the sound quality of these headphones is incredible with deep bass and clear vocals.",
    "this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.",
    "great wireless earbuds with noise cancellation that blocks out all background sound.",
    "the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.",
    "this portable charger saved me during travel and it charges my phone three times on a single charge.",
    "excellent value for money and the build quality feels premium despite the affordable price.",
    "best purchase i made this year and i would definitely buy from this brand again."
]

In [7]:
dataset = Dataset.from_dict({"text": corpus})

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

split = tokenized_dataset.train_test_split(test_size=0.2)

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

In [8]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [12]:
training_args = TrainingArguments(
    output_dir="./review_model",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    save_strategy="no",
    logging_steps=5,
    learning_rate=5e-5,
    fp16=torch.cuda.is_available()
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    data_collator=data_collator
)

In [14]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,4.597963
10,4.492310
15,3.327283
20,3.288662


TrainOutput(global_step=20, training_loss=3.9265545845031737, metrics={'train_runtime': 2.8092, 'train_samples_per_second': 12.459, 'train_steps_per_second': 7.119, 'total_flos': 571586641920.0, 'train_loss': 3.9265545845031737, 'epoch': 5.0})

In [15]:
eval_result = trainer.evaluate()
print("Perplexity:", math.exp(eval_result["eval_loss"]))

Perplexity: 37.0441739703586


In [16]:
print("\n=== AFTER FINE-TUNING ===\n")

for prompt in review_prompts:
    output = generate_text(model, tokenizer, prompt)
    print("Prompt:", prompt)
    print("Baseline:", baseline[prompt])
    print("Fine-Tuned:", output)
    print()


=== AFTER FINE-TUNING ===

Prompt: This product is
Baseline: This product is designed to be simple to use for any occasion.















































Fine-Tuned: This product is super low cost, but this one is very compact and offers a great power and it is fast enough to hold a phone.

































Prompt: I bought this phone and
Baseline: I bought this phone and I can't wait. I'll put a 5 star review on it. Will be buying the phone again in the next couple months. I really recommend it.























Fine-Tuned: I bought this phone and my husband was so impressed with it, I took it in my pocket and it took me out on my way home and even when I went to bed I found this phone in my pocket. I was very happy with the size and the phone is sturdy. I need a bit

Prompt: The quality of this item
Baseline: The quality of this item is not affected by the item and not necessarily reflect the quality of this item.

































In [17]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer2 = GPT2Tokenizer.from_pretrained("distilgpt2")
model2 = GPT2LMHeadModel.from_pretrained("distilgpt2")

tokenizer2.pad_token = tokenizer2.eos_token
model2.config.pad_token_id = tokenizer2.eos_token_id

model2.to(device)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [18]:
recipe_prompts = [
    "To make butter chicken",
    "For pasta carbonara",
    "To prepare chocolate cake"
]

baseline2 = {}

print("=== BASELINE RECIPES ===\n")

for prompt in recipe_prompts:
    text = generate_text(model2, tokenizer2, prompt)
    baseline2[prompt] = text
    print("Prompt:", prompt)
    print("Output:", text)
    print()

=== BASELINE RECIPES ===

Prompt: To make butter chicken
Output: To make butter chicken. You will need 1/2 lb of butter chicken chicken or 1/2 cup of butter chicken.

Prompt: For pasta carbonara
Output: For pasta carbonara or spaghetti. This is something that is often overlooked by most Italian pasta lovers.
If you are looking for an Italian pasta with a very light texture, you need to be able to place it in a cup of water or a warm hot water to add to the pasta.


Prompt: To prepare chocolate cake
Output: To prepare chocolate cake.

But since I had previously eaten these cookies, I really, really needed that one because I loved them so much.
This dessert, which I have a few more years to eat (and I am going to spoil a bit later on).
So I have decided



In [19]:
recipes = [
    "to make butter chicken start by marinating chicken pieces in yogurt with turmeric chili powder and garam masala for one hour.",
    "heat butter in a pan and fry onions until golden brown then add ginger garlic paste and cook for two minutes.",
    "add tomato puree and cook on low heat for ten minutes until the oil separates from the masala.",
    "add the marinated chicken and cook on medium heat for fifteen minutes until fully cooked.",
    "finish with fresh cream and kasuri methi and serve hot with naan or steamed rice.",

    "for pasta carbonara boil spaghetti in salted water until al dente and reserve half cup of pasta water.",
    "fry diced pancetta in olive oil until crispy and set aside.",
    "whisk together egg yolks parmesan cheese and black pepper in a bowl.",
    "toss the hot pasta with pancetta and remove from heat then quickly stir in the egg mixture.",
    "the residual heat will cook the eggs into a creamy sauce and serve immediately with extra parmesan.",

    "for chocolate chip cookies cream together butter and sugar until light and fluffy.",
    "beat in eggs one at a time then add vanilla extract and mix well.",
    "fold in flour baking soda and salt then gently stir in chocolate chips.",
    "scoop dough onto a baking sheet and bake at 180 degrees for twelve minutes until golden.",
    "let cookies cool on the tray for five minutes before transferring to a wire rack."
]

In [20]:
dataset2 = Dataset.from_dict({"text": recipes})

def tokenize_recipe(examples):
    return tokenizer2(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

tokenized_dataset2 = dataset2.map(tokenize_recipe, batched=True)
tokenized_dataset2 = tokenized_dataset2.remove_columns(["text"])

split2 = tokenized_dataset2.train_test_split(test_size=0.2)

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

In [21]:
collator2 = DataCollatorForLanguageModeling(
    tokenizer=tokenizer2,
    mlm=False
)

In [24]:
training_args2 = TrainingArguments(
    output_dir="./recipe_model",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    save_strategy="no",
    logging_steps=5,
    learning_rate=5e-5,
    fp16=torch.cuda.is_available()
)

In [25]:
trainer2 = Trainer(
    model=model2,
    args=training_args2,
    train_dataset=split2["train"],
    eval_dataset=split2["test"],
    data_collator=collator2
)

In [26]:
trainer2.train()

Step,Training Loss
5,4.177651
10,4.025881
15,3.262423
20,2.710777
25,2.889902
30,2.623104


TrainOutput(global_step=30, training_loss=3.281622854868571, metrics={'train_runtime': 2.8816, 'train_samples_per_second': 20.822, 'train_steps_per_second': 10.411, 'total_flos': 979862814720.0, 'train_loss': 3.281622854868571, 'epoch': 5.0})

In [27]:
eval_result2 = trainer2.evaluate()
print("Perplexity:", math.exp(eval_result2["eval_loss"]))

Perplexity: 31.374922291048676


In [28]:
print("\n=== AFTER FINE-TUNING ===\n")

for prompt in recipe_prompts:
    output = generate_text(model2, tokenizer2, prompt)
    print("Prompt:", prompt)
    print("Baseline:", baseline2[prompt])
    print("Fine-Tuned:", output)
    print()


=== AFTER FINE-TUNING ===

Prompt: To make butter chicken
Baseline: To make butter chicken. You will need 1/2 lb of butter chicken chicken or 1/2 cup of butter chicken.
Fine-Tuned: To make butter chicken in a pan and cook until fragrant with salt and pepper. To serve, fry chicken in pan for ten minutes before flipping off the skillet. Remove from heat and cook for ten minutes before removing from heat. Remove from heat and cook for twenty minutes. In a bowl of olive

Prompt: For pasta carbonara
Baseline: For pasta carbonara or spaghetti. This is something that is often overlooked by most Italian pasta lovers.
If you are looking for an Italian pasta with a very light texture, you need to be able to place it in a cup of water or a warm hot water to add to the pasta.

Fine-Tuned: For pasta carbonara, heat the onions in a pan and add garlic paste and cook for an hour. Add masala and chilli. Add salt and cook for 1 hour. Add masala and chilli. Add water. Cook for 2 minutes. Slowly add masa